In [ ]:
!pip install -q "transformers==4.45.2" timm albumentations scikit-learn pandas numpy torch networkx

In [ ]:
import cv2
import numpy as np
from PIL import Image


def ensure_uint8_rgb(img_rgb: np.ndarray) -> np.ndarray:
    """
    Ensure image is RGB uint8.
    """
    if img_rgb.dtype != np.uint8:
        img_rgb = np.clip(img_rgb, 0, 255).astype(np.uint8)
    return img_rgb


def apply_gamma_rgb(img_rgb: np.ndarray, gamma: float = 1.0) -> np.ndarray:
    """
    Apply gamma correction using a LUT.
    Convention here:
        gamma > 1.0  -> brighten
        gamma < 1.0  -> darken
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    if gamma <= 0:
        raise ValueError("gamma must be > 0")

    inv_gamma = 1.0 / gamma
    table = np.array(
        [((i / 255.0) ** inv_gamma) * 255 for i in range(256)],
        dtype=np.uint8
    )
    return cv2.LUT(img_rgb, table)


def unsharp_mask_rgb(
    img_rgb: np.ndarray,
    sigma: float = 1.0,
    amount: float = 0.8
) -> np.ndarray:
    """
    Mild sharpening using unsharp masking.
    More stable than a hard filter2D sharpening kernel.
    """
    img_rgb = ensure_uint8_rgb(img_rgb)
    blurred = cv2.GaussianBlur(img_rgb, (0, 0), sigmaX=sigma, sigmaY=sigma)
    sharpened = cv2.addWeighted(img_rgb, 1.0 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def lynx_preprocessing(
    img_rgb: np.ndarray,
    clahe_clip: float = 2.0,
    clahe_grid: tuple = (4, 4),
    sharpen_amount: float = 0.6,
    denoise_sigma_color: int = 35,
    denoise_sigma_space: int = 35,
) -> np.ndarray:
    """
    Refined preprocessing for lynx:
    - mild local contrast enhancement on luminance
    - mild sharpening
    - mild bilateral denoising

    Input: RGB uint8
    Output: RGB uint8
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clahe_clip,
        tileGridSize=clahe_grid
    )
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    img_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    img_rgb = unsharp_mask_rgb(img_rgb, sigma=1.0, amount=sharpen_amount)

    img_rgb = cv2.bilateralFilter(
        img_rgb,
        d=5,
        sigmaColor=denoise_sigma_color,
        sigmaSpace=denoise_sigma_space
    )

    return img_rgb


def turtle_preprocessing(
    img_rgb: np.ndarray,
    red_boost: float = 1.12,
    gamma: float = 1.15,
    clahe_clip: float = 1.8,
    clahe_grid: tuple = (4, 4),
) -> np.ndarray:
    """
    Refined turtle preprocessing:
    - mild red-channel compensation
    - mild contrast enhancement
    - slight brightening gamma

    Input: RGB uint8
    Output: RGB uint8
    """
    img_rgb = ensure_uint8_rgb(img_rgb).astype(np.float32)

    img_rgb[..., 0] *= red_boost
    img_rgb = np.clip(img_rgb, 0, 255).astype(np.uint8)

    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clahe_clip,
        tileGridSize=clahe_grid
    )
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    img_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    img_rgb = apply_gamma_rgb(img_rgb, gamma=gamma)

    return img_rgb


def salamander_preprocessing(
    img_rgb: np.ndarray,
    gamma: float = 1.12,
    edge_weight: float = 0.20,
    blur_before_edges: bool = True,
) -> np.ndarray:
    """
    Refined salamander preprocessing:
    - safe Sobel edge extraction
    - gentle edge overlay
    - mild gamma brightening

    Input: RGB uint8
    Output: RGB uint8
    """
    img_rgb = ensure_uint8_rgb(img_rgb)

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    if blur_before_edges:
        gray = cv2.GaussianBlur(gray, (3, 3), 0)

    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    edges = cv2.magnitude(sobelx, sobely)

    max_val = float(edges.max())
    if max_val > 1e-6:
        edges = (edges / max_val) * 255.0
    else:
        edges = np.zeros_like(edges, dtype=np.float32)

    edges = edges.astype(np.uint8)
    edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

    base_weight = 1.0 - edge_weight
    img_rgb = cv2.addWeighted(img_rgb, base_weight, edges_rgb, edge_weight, 0)

    img_rgb = apply_gamma_rgb(img_rgb, gamma=gamma)

    return img_rgb

In [ ]:
import os
from typing import Optional, Callable, Dict, Any

import pandas as pd
import numpy as np
from PIL import Image

from torch.utils.data import Dataset


def default_species_preprocess(img_rgb: np.ndarray, dataset_name: str) -> np.ndarray:
    """
    Dispatch species-specific preprocessing.
    Input / output: RGB uint8 numpy array
    """
    dataset_name = str(dataset_name).lower()

    if "lynx" in dataset_name:
        return lynx_preprocessing(img_rgb)
    elif "turtle" in dataset_name:
        return turtle_preprocessing(img_rgb)
    elif "salamander" in dataset_name:
        return salamander_preprocessing(img_rgb)
    else:
        return img_rgb


class OpenSetAnimalDataset(Dataset):
    def __init__(
        self,
        csv_file: str,
        root_dir: str = "",
        transform: Optional[Callable] = None,
        preprocess_fn: Optional[Callable[[np.ndarray, str], np.ndarray]] = None,
        label_to_idx: Optional[Dict[str, int]] = None,
        training: bool = True,
        use_proc_path: bool = False,
        return_path: bool = False,
    ):
        self.df = pd.read_csv(csv_file).copy()
        self.root_dir = root_dir
        self.transform = transform
        self.preprocess_fn = preprocess_fn
        self.training = training
        self.return_path = return_path

        required_cols = {"dataset", "image_id", "identity"}
        missing = required_cols - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing required columns in {csv_file}: {missing}")

        if use_proc_path and "proc_path" in self.df.columns:
            self.path_col = "proc_path"
        elif "path" in self.df.columns:
            self.path_col = "path"
        else:
            raise ValueError("CSV must contain 'path' or 'proc_path'.")

        self.df = self.df[self.df[self.path_col].notna()].reset_index(drop=True)

        # species-aware identity key
        self.df["id_key"] = (
            self.df["dataset"].astype(str) + "::" + self.df["identity"].astype(str)
        )

        if label_to_idx is None and training:
            identities = sorted(self.df["id_key"].unique().tolist())
            self.label_to_idx = {identity: idx for idx, identity in enumerate(identities)}
        else:
            self.label_to_idx = label_to_idx

        self.labels = []
        self.is_known_list = []

        for _, row in self.df.iterrows():
            id_key = str(row["id_key"])
            if self.label_to_idx is not None and id_key in self.label_to_idx:
                self.labels.append(self.label_to_idx[id_key])
                self.is_known_list.append(True)
            else:
                self.labels.append(-1)
                self.is_known_list.append(False)

    def __len__(self) -> int:
        return len(self.df)

    def _resolve_path(self, rel_path: str) -> str:
        rel_path = str(rel_path)
        if os.path.isabs(rel_path):
            return rel_path
        return os.path.join(self.root_dir, rel_path) if self.root_dir else rel_path

    def _load_image(self, img_path: str) -> Image.Image:
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")
        return Image.open(img_path).convert("RGB")

    def __getitem__(self, index: int) -> Dict[str, Any]:
        row = self.df.iloc[index]

        rel_path = row[self.path_col]
        full_path = self._resolve_path(rel_path)

        img = self._load_image(full_path)
        img_rgb = np.array(img)

        dataset_name = str(row["dataset"])

        if self.preprocess_fn is not None:
            img_rgb = self.preprocess_fn(img_rgb, dataset_name)
            img = Image.fromarray(img_rgb)

        if self.transform is not None:
            img = self.transform(img)

        sample = {
            "image": img,
            "label": int(self.labels[index]),
            "identity": str(row["identity"]),
            "id_key": str(row["id_key"]),
            "dataset": dataset_name,
            "image_id": row["image_id"],
            "is_known": bool(self.is_known_list[index]),
        }

        if "val_type" in row.index:
            sample["val_type"] = str(row["val_type"])

        if self.return_path:
            sample["path"] = full_path

        return sample

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((440, 440)),
    transforms.RandomRotation(degrees=35),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = OpenSetAnimalDataset(
    csv_file="fine_tune_train.csv",
    root_dir="Dataset Root Path",   
    transform=train_transform,
    preprocess_fn=default_species_preprocess,
    training=True,
    use_proc_path=False,
    return_path=True,
)

print("num train samples:", len(train_dataset))
print("num train classes:", len(train_dataset.label_to_idx))
sample = train_dataset[0]
print(sample["image"].shape, sample["label"], sample["identity"], sample["dataset"])

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((440, 440)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_known_dataset = OpenSetAnimalDataset(
    csv_file="val_known.csv",
    root_dir="Dataset Root Path",
    transform=val_transform,
    preprocess_fn=default_species_preprocess,
    label_to_idx=train_dataset.label_to_idx,
    training=False,
    use_proc_path=False,
    return_path=True,
)

sample = val_known_dataset[0]
print(sample["label"], sample["is_known"], sample["val_type"])

In [ ]:
val_unknown_dataset = OpenSetAnimalDataset(
    csv_file="val_unknown.csv",
    root_dir="Dataset Root Path",
    transform=val_transform,
    preprocess_fn=default_species_preprocess,
    label_to_idx=train_dataset.label_to_idx,
    training=False,
    use_proc_path=False,
    return_path=True,
)

sample = val_unknown_dataset[0]
print(sample["label"], sample["is_known"], sample["val_type"])   

In [ ]:
import numpy as np
from collections import defaultdict
from typing import Dict, List, Sequence, Union, Optional

import torch
from torch.utils.data import Sampler


class SpeciesBalancedPKBatchSampler(Sampler[List[int]]):
    """
    Species-balanced P x K batch sampler for animal re-id.

    Each batch contains:
        - P total identities
        - K samples per identity
        - species-balanced identity selection across species

    Total batch size = P * K

    Requirements for data_source:
        - data_source.labels : list[int]
        - data_source.df[species_col] : species/dataset column
          (for your case, species_col='dataset')

    Unknown labels (-1) are ignored automatically.
    """

    def __init__(
        self,
        data_source,
        p: int,
        k: int,
        species_col: str = "dataset",
        species_per_batch: Optional[Dict[str, int]] = None,
        num_batches: Optional[int] = None,
        drop_last: bool = True,
        seed: int = 42,
    ):
        """
        Args:
            data_source:
                dataset object with:
                    - .labels
                    - .df
            p:
                total number of identities per batch
            k:
                number of instances per identity
            species_col:
                column in dataset.df that indicates species/dataset
            species_per_batch:
                optional fixed per-batch species identity counts
                example:
                    {"LynxID2025": 4, "SalamanderID2025": 4, "SeaTurtleID2022": 4}
                Sum must equal p.
                If None, it is auto-balanced.
            num_batches:
                batches per epoch; if None, estimated automatically
            drop_last:
                used only for automatic epoch length estimate
            seed:
                random seed base
        """
        super().__init__()

        if not hasattr(data_source, "labels"):
            raise ValueError("data_source must have attribute .labels")
        if not hasattr(data_source, "df"):
            raise ValueError("data_source must have attribute .df")
        if species_col not in data_source.df.columns:
            raise ValueError(f"species_col='{species_col}' not found in dataset.df")

        if p <= 0 or k <= 0:
            raise ValueError("p and k must be positive integers")

        self.data_source = data_source
        self.labels = [int(x) for x in data_source.labels]
        self.df = data_source.df.reset_index(drop=True)
        self.species_col = species_col
        self.p = int(p)
        self.k = int(k)
        self.batch_size = self.p * self.k
        self.drop_last = drop_last
        self.base_seed = int(seed)
        self.epoch = 0

        self.label_to_indices = self._build_label_to_indices(self.labels)
        self.label_to_species = self._build_label_to_species(self.labels, self.df, self.species_col)
        self.species_to_labels = self._build_species_to_labels(self.label_to_species, self.label_to_indices)

        # only valid train labels
        self.valid_labels = sorted([lab for lab in self.label_to_indices if lab >= 0])
        if len(self.valid_labels) < self.p:
            raise ValueError(
                f"Not enough unique valid identities ({len(self.valid_labels)}) for p={self.p}"
            )

        self.valid_species = sorted([sp for sp, labs in self.species_to_labels.items() if len(labs) > 0])
        if len(self.valid_species) == 0:
            raise ValueError("No valid species found.")

        # species counts per batch
        if species_per_batch is None:
            self.species_per_batch = self._auto_species_per_batch()
        else:
            self.species_per_batch = dict(species_per_batch)
            if sum(self.species_per_batch.values()) != self.p:
                raise ValueError("Sum of species_per_batch values must equal p.")
            for sp, cnt in self.species_per_batch.items():
                if sp not in self.species_to_labels:
                    raise ValueError(f"Species '{sp}' not found in training set.")
                if cnt < 0:
                    raise ValueError(f"species_per_batch['{sp}'] must be >= 0")

        self.num_samples = sum(len(self.label_to_indices[lab]) for lab in self.valid_labels)
        if num_batches is None:
            if self.drop_last:
                self.num_batches = max(1, self.num_samples // self.batch_size)
            else:
                self.num_batches = max(1, int(np.ceil(self.num_samples / self.batch_size)))
        else:
            self.num_batches = int(num_batches)

    def _build_label_to_indices(self, labels: Sequence[int]) -> Dict[int, List[int]]:
        label_to_indices = defaultdict(list)
        for idx, lab in enumerate(labels):
            label_to_indices[int(lab)].append(idx)
        return dict(label_to_indices)

    def _build_label_to_species(self, labels, df, species_col) -> Dict[int, str]:
        """
        Map each label -> species.
        Assumes one label belongs to exactly one species.
        """
        label_to_species = {}
        for idx, lab in enumerate(labels):
            if lab < 0:
                continue
            species = str(df.iloc[idx][species_col])
            if lab in label_to_species:
                if label_to_species[lab] != species:
                    raise ValueError(
                        f"Label {lab} appears in multiple species: "
                        f"{label_to_species[lab]} and {species}. "
                        f"Use species-aware identity keys in the dataset class."
                    )
            else:
                label_to_species[lab] = species
        return label_to_species

    def _build_species_to_labels(self, label_to_species, label_to_indices):
        species_to_labels = defaultdict(list)
        for lab, sp in label_to_species.items():
            if lab >= 0 and len(label_to_indices.get(lab, [])) > 0:
                species_to_labels[sp].append(lab)
        for sp in species_to_labels:
            species_to_labels[sp] = sorted(species_to_labels[sp])
        return dict(species_to_labels)

    def _auto_species_per_batch(self) -> Dict[str, int]:
        """
        Evenly distribute P identities across species.
        Remainder goes to species with more available identities.
        """
        species_counts = {sp: len(labs) for sp, labs in self.species_to_labels.items() if len(labs) > 0}
        species = sorted(species_counts.keys())
        n_species = len(species)

        base = self.p // n_species
        rem = self.p % n_species

        allocation = {sp: base for sp in species}

        # give remainder to species with more identities
        species_sorted = sorted(species, key=lambda s: species_counts[s], reverse=True)
        for i in range(rem):
            allocation[species_sorted[i]] += 1

        return allocation

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return self.num_batches

    def _sample_k_for_label(
        self,
        label: int,
        rng: np.random.Generator,
        per_label_pools: Dict[int, List[int]],
        per_label_pos: Dict[int, int],
    ) -> List[int]:
        """
        Sample K indices for one identity.
        """
        idxs = self.label_to_indices[label]

        if len(idxs) >= self.k:
            pool = per_label_pools[label]
            pos = per_label_pos[label]

            if pos + self.k <= len(pool):
                out = pool[pos:pos + self.k]
                per_label_pos[label] += self.k
                return out

            remaining = pool[pos:]
            needed = self.k - len(remaining)

            new_pool = rng.permutation(idxs).tolist()
            out = remaining + new_pool[:needed]

            per_label_pools[label] = new_pool
            per_label_pos[label] = needed
            return out

        # sparse identity -> with replacement
        return rng.choice(idxs, size=self.k, replace=True).tolist()

    def _choose_species_balanced_labels(self, rng: np.random.Generator) -> List[int]:
        """
        Choose P unique labels with species balancing.
        """
        chosen = []
        chosen_set = set()

        # 1) satisfy requested per-species quotas as much as possible
        for species, quota in self.species_per_batch.items():
            if quota <= 0:
                continue

            candidates = [lab for lab in self.species_to_labels.get(species, []) if lab not in chosen_set]

            if len(candidates) >= quota:
                picked = rng.choice(candidates, size=quota, replace=False).tolist()
            else:
                picked = list(candidates)

            chosen.extend(picked)
            chosen_set.update(picked)

        # 2) fill any deficit from remaining labels globally
        deficit = self.p - len(chosen)
        if deficit > 0:
            remaining = [lab for lab in self.valid_labels if lab not in chosen_set]
            if len(remaining) < deficit:
                raise ValueError(
                    f"Not enough remaining unique labels to fill batch deficit={deficit}. "
                    f"Try smaller p."
                )
            extra = rng.choice(remaining, size=deficit, replace=False).tolist()
            chosen.extend(extra)
            chosen_set.update(extra)

        return chosen

    def __iter__(self):
        rng = np.random.default_rng(self.base_seed + self.epoch)

        per_label_pools = {
            lab: rng.permutation(self.label_to_indices[lab]).tolist()
            for lab in self.valid_labels
        }
        per_label_pos = {lab: 0 for lab in self.valid_labels}

        for _ in range(self.num_batches):
            batch_labels = self._choose_species_balanced_labels(rng)

            batch_indices = []
            for lab in batch_labels:
                batch_indices.extend(
                    self._sample_k_for_label(
                        label=lab,
                        rng=rng,
                        per_label_pools=per_label_pools,
                        per_label_pos=per_label_pos,
                    )
                )

            rng.shuffle(batch_indices)
            yield batch_indices

In [ ]:
from torch.utils.data import DataLoader

train_sampler = SpeciesBalancedPKBatchSampler(
    data_source=train_dataset,
    p=12,
    k=2,
    species_col="dataset",
    species_per_batch={
        "LynxID2025": 4,
        "SalamanderID2025": 4,
        "SeaTurtleID2022": 4,
    },
    num_batches=250,
    seed=42,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
from torch.utils.data import DataLoader

train_sampler = SpeciesBalancedPKBatchSampler(
    data_source=train_dataset,
    p=12,
    k=2,
    species_col="dataset",
    species_per_batch={
        "LynxID2025": 4,
        "SalamanderID2025": 4,
        "SeaTurtleID2022": 4,
    },
    num_batches=250,
    seed=42,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
val_known_loader = DataLoader(
    val_known_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

val_unknown_loader = DataLoader(
    val_unknown_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()

    all_embs = []
    all_labels = []
    all_identities = []
    all_datasets = []
    all_known_flags = []

    for batch in loader:
        images = batch["image"].to(device)
        emb = model(images)

        all_embs.append(emb.cpu())
        all_labels.extend(batch["label"].tolist())
        all_identities.extend(batch["identity"])
        all_datasets.extend(batch["dataset"])
        all_known_flags.extend(batch["is_known"])

    all_embs = torch.cat(all_embs, dim=0)
    return {
        "embeddings": all_embs,
        "labels": all_labels,
        "identities": all_identities,
        "datasets": all_datasets,
        "is_known": all_known_flags,
    }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel


MODEL_ID = "conservationxlabs/miewid-msv3"


def load_miewid_hf(model_id=MODEL_ID):
    config = AutoConfig.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    return config, model

In [ ]:
class MiewIDBackboneHF(nn.Module):
    """
    Backbone wrapper for the Hugging Face custom MiewID model.
    Produces feature vectors only.
    """
    def __init__(self, hf_model, l2_norm=False):
        super().__init__()
        self.hf_model = hf_model
        self.l2_norm = l2_norm

    def forward(self, x):
        # 1) best case: model exposes forward_features
        if hasattr(self.hf_model, "forward_features"):
            feats = self.hf_model.forward_features(x)

        # 2) some custom models expose extract_features
        elif hasattr(self.hf_model, "extract_features"):
            feats = self.hf_model.extract_features(x)

        # 3) fallback: normal forward
        else:
            out = self.hf_model(x)

            if isinstance(out, torch.Tensor):
                feats = out
            elif isinstance(out, (tuple, list)):
                feats = out[0]
            elif isinstance(out, dict):
                if "embeddings" in out:
                    feats = out["embeddings"]
                elif "features" in out:
                    feats = out["features"]
                elif "logits" in out:
                    feats = out["logits"]
                elif "last_hidden_state" in out:
                    feats = out["last_hidden_state"]
                else:
                    feats = list(out.values())[0]
            else:
                # handle custom output objects
                if hasattr(out, "embeddings"):
                    feats = out.embeddings
                elif hasattr(out, "features"):
                    feats = out.features
                elif hasattr(out, "logits"):
                    feats = out.logits
                elif hasattr(out, "last_hidden_state"):
                    feats = out.last_hidden_state
                else:
                    raise TypeError(f"Unsupported MiewID output type: {type(out)}")

        if feats.ndim > 2:
            feats = torch.flatten(feats, 1)

        if self.l2_norm:
            feats = F.normalize(feats, p=2, dim=1)

        return feats

In [ ]:
@torch.no_grad()
def infer_feat_dim(backbone, image_size=440, device="cuda"):
    backbone.eval().to(device)
    x = torch.randn(2, 3, image_size, image_size, device=device)
    y = backbone(x)
    if y.ndim > 2:
        y = torch.flatten(y, 1)
    return y.shape[1]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SphereFace2FocalHead(nn.Module):
    """
    SphereFace2 head with optional focal loss.

    Fixes:
    1) No detached / no_grad margin modification
    2) Weight normalization uses F.normalize(self.weight, ...)
    3) True differentiable focal weighting
    4) Clean backbone-head style API

    magn_type:
        'C' -> SphereFace2-C
        'A' -> SphereFace2-A
        'M' -> SphereFace2-M
    """

    def __init__(
        self,
        in_features,
        num_classes,
        magn_type="C",
        alpha=0.7,
        r=30.0,
        m=0.25,
        t=2.0,
        lw=20.0,
        focal_gamma=1.0,
        dropout=0.0,
        use_focal=True,
        eps=1e-20,
    ):
        super().__init__()

        self.in_features = in_features
        self.num_classes = num_classes
        self.magn_type = magn_type
        self.alpha = alpha
        self.r = r
        self.m = m
        self.t = t
        self.lw = lw
        self.focal_gamma = focal_gamma
        self.use_focal = use_focal
        self.eps = eps

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Standard classifier weight shape: [num_classes, in_features]
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_normal_(self.weight)

        # SphereFace2 bias initialization
        z = alpha / ((1.0 - alpha) * (num_classes - 1.0))

        if magn_type == "C":
            ay = r * (2.0 * (0.5 ** t) - 1.0 - m)
            ai = r * (2.0 * (0.5 ** t) - 1.0 + m)

        elif magn_type == "A":
            theta_y = min(math.pi, math.pi / 2.0 + m)
            ay = r * (2.0 * (((math.cos(theta_y) + 1.0) / 2.0) ** t) - 1.0)
            ai = r * (2.0 * (0.5 ** t) - 1.0)

        elif magn_type == "M":
            theta_y = min(math.pi, m * math.pi / 2.0)
            ay = r * (2.0 * (((math.cos(theta_y) + 1.0) / 2.0) ** t) - 1.0)
            ai = r * (2.0 * (0.5 ** t) - 1.0)

        else:
            raise NotImplementedError(f"Unknown magn_type={magn_type}")

        temp = (1.0 - z) ** 2 + 4.0 * z * math.exp(ay - ai)
        b_init = math.log(2.0 * z) - ai - math.log(1.0 - z + math.sqrt(temp))

        self.b = nn.Parameter(torch.tensor([b_init], dtype=torch.float32))

    def _normalized_cosine(self, feats):
        """
        Cosine similarity between normalized features and normalized class weights.
        """
        feats = self.dropout(feats)
        x = F.normalize(feats, p=2, dim=1)
        w = F.normalize(self.weight, p=2, dim=1)
        cos_theta = F.linear(x, w)  # [B, C]
        cos_theta = cos_theta.clamp(-1.0 + self.eps, 1.0 - self.eps)
        return cos_theta

    def predict_logits(self, feats):
        """
        Plain logits without training margin modification.
        Useful for monitoring / inference.
        """
        cos_theta = self._normalized_cosine(feats)
        logits = self.r * cos_theta + self.b
        return logits

    def _margin_logits(self, feats, labels):
        """
        Margin-modified logits used during training.
        Fully differentiable.
        """
        cos_theta = self._normalized_cosine(feats)  # [B, C]

        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)

        if self.magn_type == "C":
            # Differentiable C-type transform
            g_cos_theta = 2.0 * ((cos_theta + 1.0) / 2.0).pow(self.t) - 1.0
            g_cos_theta = g_cos_theta - self.m * (2.0 * one_hot - 1.0)

        elif self.magn_type == "A":
            # Add angular margin only to the target class
            theta = torch.acos(cos_theta)  # [B, C]
            theta_target = theta.gather(1, labels.view(-1, 1)) + self.m
            theta_target = theta_target.clamp(self.eps, math.pi - self.eps)

            theta_mod = theta.clone()
            theta_mod.scatter_(1, labels.view(-1, 1), theta_target)

            g_cos_theta = torch.cos(theta_mod)
            g_cos_theta = 2.0 * ((g_cos_theta + 1.0) / 2.0).pow(self.t) - 1.0

        elif self.magn_type == "M":
            # Multiply target angle by m
            theta = torch.acos(cos_theta)  # [B, C]
            theta_target = theta.gather(1, labels.view(-1, 1)) * self.m
            theta_target = theta_target.clamp(self.eps, math.pi - self.eps)

            theta_mod = theta.clone()
            theta_mod.scatter_(1, labels.view(-1, 1), theta_target)

            g_cos_theta = torch.cos(theta_mod)
            g_cos_theta = 2.0 * ((g_cos_theta + 1.0) / 2.0).pow(self.t) - 1.0

        else:
            raise NotImplementedError(f"Unknown magn_type={self.magn_type}")

        logits = self.r * g_cos_theta + self.b
        return logits, one_hot

    def compute_loss(self, feats, labels, return_logits=False):
        """
        SphereFace2 + optional focal loss.
        """
        logits, one_hot = self._margin_logits(feats, labels)

        # SphereFace2 balancing weight
        sphere_weight = self.alpha * one_hot + (1.0 - self.alpha) * (1.0 - one_hot)
        sphere_weight = self.lw * self.num_classes / self.r * sphere_weight

        # BCE per element
        bce = F.binary_cross_entropy_with_logits(
            logits,
            one_hot,
            reduction="none",
        )

        if self.use_focal:
            probs = torch.sigmoid(logits)
            pt = one_hot * probs + (1.0 - one_hot) * (1.0 - probs)
            focal_weight = (1.0 - pt).pow(self.focal_gamma)
        else:
            focal_weight = 1.0

        loss_matrix = sphere_weight * focal_weight * bce
        loss = loss_matrix.mean()

        if return_logits:
            # For train top-k logging, plain logits are more interpretable
            pred_logits = self.predict_logits(feats)
            return loss, pred_logits

        return loss

    def forward(self, feats, labels=None):
        """
        Head-style forward:
        return plain logits by default.
        """
        return self.predict_logits(feats)

In [ ]:
class ReIDModel(nn.Module):
    def __init__(self, backbone, head, normalize_feats=False):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.normalize_feats = normalize_feats

    def forward_features(self, x):
        feats = self.backbone(x)
        if self.normalize_feats:
            feats = F.normalize(feats, p=2, dim=1)
        return feats

    def forward(self, x, labels=None, return_features=False):
        feats = self.forward_features(x)
        logits = self.head(feats, labels)

        if return_features:
            return logits, F.normalize(feats, p=2, dim=1)
        return logits

In [ ]:
def build_miewid_linear_model_hf(
    num_classes,
    model_id="conservationxlabs/miewid-msv3",
    dropout=0.1,
    image_size=440,
    device="cuda",
):
    config, hf_model = load_miewid_hf(model_id)

    backbone = MiewIDBackboneHF(hf_model, l2_norm=False)
    feat_dim = infer_feat_dim(backbone, image_size=image_size, device=device)

    head = SphereFace2FocalHead(
                in_features=feat_dim,
                num_classes=num_classes,
                magn_type="A",      
                alpha=0.99,
                r=55,
                m=0.3,
                t=3.0,
                lw=20,
                focal_gamma=1.0,
                dropout=0.1,
                use_focal=True,
            )

    model = ReIDModel(
        backbone=backbone,
        head=head,
        normalize_feats=False,
    )
    return config, model, feat_dim

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = len(train_dataset.label_to_idx)

config, model, feat_dim = build_miewid_linear_model_hf(
    num_classes=num_classes,
    model_id="conservationxlabs/miewid-msv3",
    dropout=0.1,
    image_size=440,
    device=device,
)

model = model.to(device)

print(config)
print("feature dim:", feat_dim)
print("num classes:", num_classes)

In [ ]:
import re
def freeze_module(module):
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True
def unfreeze_last_blocks(backbone, last_n_blocks=2, verbose=True):
    """
    Freeze all backbone params, then unfreeze only the last N blocks
    plus common final layers.
    """
    freeze_module(backbone)

    block_ids = []

    # collect block indices
    for name, _ in backbone.hf_model.named_parameters():
        m = re.search(r"blocks\.(\d+)", name)
        if m:
            block_ids.append(int(m.group(1)))

    block_ids = sorted(set(block_ids))
    if len(block_ids) == 0:
        raise ValueError("No blocks.X pattern found. Print parameter names first.")

    target_blocks = set(block_ids[-last_n_blocks:])

    matched = []
    for name, p in backbone.hf_model.named_parameters():
        lname = name.lower()

        block_match = re.search(r"blocks\.(\d+)", name)
        in_last_block = block_match and int(block_match.group(1)) in target_blocks

        final_layer = any(x in lname for x in [
            "conv_head", "bn2", "head", "proj", "projection", "embedding"
        ])

        if in_last_block or final_layer:
            p.requires_grad = True
            matched.append(name)

    if verbose:
        print("Target blocks:", sorted(target_blocks))
        print("Unfrozen params:")
        for n in matched:
            print("  ", n)

    return matched

freeze_module(model.backbone)
matched = unfreeze_last_blocks(model.backbone, last_n_blocks=2, verbose= False)
unfreeze_module(model.head)

In [ ]:
optimizer = torch.optim.AdamW(
    [
        {
            "params": [p for p in model.backbone.parameters() if p.requires_grad],
            "lr": 1e-5,
        },
        {
            "params": [p for p in model.head.parameters() if p.requires_grad],
            "lr": 1e-3,
        },
    ],
    weight_decay=1e-4,
)

In [ ]:
import os
import math
import time
import json
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm


# -----------------------------
# Average meter
# -----------------------------
class AverageMeter:
    def __init__(self, name="meter"):
        self.name = name
        self.reset()

    def reset(self):
        self.val = 0.0
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0

    def update(self, val, n=1):
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


# -----------------------------
# Top-k classification accuracy
# -----------------------------
@torch.no_grad()
def topk_accuracy_from_logits(logits, targets, topk=(1, 5, 20)):
    """
    logits: [B, C]
    targets: [B]
    returns dict {k: acc_percent}
    """
    maxk = min(max(topk), logits.size(1))
    batch_size = targets.size(0)

    _, pred = torch.topk(logits, k=maxk, dim=1, largest=True, sorted=True)  # [B, maxk]
    pred = pred.t()  # [maxk, B]
    correct = pred.eq(targets.view(1, -1).expand_as(pred))

    out = {}
    for k in topk:
        kk = min(k, logits.size(1))
        correct_k = correct[:kk].reshape(-1).float().sum(0)
        out[k] = float(correct_k.mul_(100.0 / batch_size).item())
    return out


# -----------------------------
# LR helpers
# -----------------------------
def get_optimizer_lrs(optimizer):
    lrs = []
    for i, group in enumerate(optimizer.param_groups):
        lrs.append(group["lr"])
    return lrs


def format_lrs(optimizer):
    lrs = get_optimizer_lrs(optimizer)
    return " | ".join([f"lr[{i}]={lr:.2e}" for i, lr in enumerate(lrs)])


# -----------------------------
# Trainable parameter summary
# -----------------------------
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# -----------------------------
# Gradient norm
# -----------------------------
def compute_grad_norm(parameters, norm_type=2.0):
    params = [p for p in parameters if p.grad is not None]
    if len(params) == 0:
        return 0.0

    norm_type = float(norm_type)
    total_norm = 0.0
    for p in params:
        param_norm = p.grad.data.norm(norm_type)
        total_norm += param_norm.item() ** norm_type
    total_norm = total_norm ** (1.0 / norm_type)
    return float(total_norm)


# -----------------------------
# Feature extraction
# -----------------------------
@torch.no_grad()
def extract_features(model, loader, device, desc="Extract features"):
    model.eval()

    all_feats = []
    all_labels = []
    all_identities = []
    all_datasets = []
    all_is_known = []
    all_image_ids = []
    all_paths = []

    progress = tqdm(loader, desc=desc, leave=False)

    for batch in progress:
        images = batch["image"].to(device, non_blocking=True)
        feats = model.forward_features(images)
        feats = F.normalize(feats, p=2, dim=1)

        all_feats.append(feats.cpu())
        all_labels.extend(batch["label"].cpu().tolist())
        all_identities.extend(list(batch["identity"]))
        all_datasets.extend(list(batch["dataset"]))
        all_is_known.extend(list(batch["is_known"]))
        all_image_ids.extend(list(batch["image_id"]))

        if "path" in batch:
            all_paths.extend(list(batch["path"]))
        else:
            all_paths.extend([None] * len(batch["identity"]))

    all_feats = torch.cat(all_feats, dim=0)

    return {
        "features": all_feats,             # [N, D]
        "labels": all_labels,              # list[int]
        "identities": all_identities,      # list[str]
        "datasets": all_datasets,          # list[str]
        "is_known": all_is_known,          # list[bool]
        "image_ids": all_image_ids,        # list
        "paths": all_paths,                # list[str|None]
    }

In [ ]:
@torch.no_grad()
def evaluate_known_retrieval_topk(
    gallery_dict,
    query_dict,
    topk=(1, 5, 20),
):
    """
    Evaluate retrieval top-k on known validation queries.

    gallery_dict, query_dict: output of extract_features()

    Important:
    - Only same-species gallery images are used for each query.
    - Query identities are compared against gallery identities.
    """
    gallery_feats = gallery_dict["features"]              # [Ng, D]
    gallery_ids = np.array(gallery_dict["identities"])
    gallery_species = np.array(gallery_dict["datasets"])

    query_feats = query_dict["features"]                  # [Nq, D]
    query_ids = np.array(query_dict["identities"])
    query_species = np.array(query_dict["datasets"])

    topk = sorted(topk)
    results = {k: 0 for k in topk}
    n_queries = len(query_ids)

    # per-species accounting
    species_results = defaultdict(lambda: {"count": 0, **{f"top{k}": 0 for k in topk}})

    # pre-index gallery by species
    species_to_gallery_idx = defaultdict(list)
    for i, sp in enumerate(gallery_species):
        species_to_gallery_idx[sp].append(i)

    for q_idx in range(n_queries):
        q_feat = query_feats[q_idx:q_idx + 1]   # [1, D]
        q_id = query_ids[q_idx]
        q_sp = query_species[q_idx]

        g_idx = species_to_gallery_idx[q_sp]
        if len(g_idx) == 0:
            continue

        g_idx = np.array(g_idx)
        g_feats = gallery_feats[g_idx]              # [M, D]
        g_ids = gallery_ids[g_idx]                  # [M]

        sims = torch.matmul(q_feat, g_feats.T).squeeze(0)   # [M]
        ranking = torch.argsort(sims, descending=True)

        ranked_ids = g_ids[ranking.cpu().numpy()].tolist()

        species_results[q_sp]["count"] += 1

        for k in topk:
            kk = min(k, len(ranked_ids))
            hit = int(q_id in ranked_ids[:kk])
            results[k] += hit
            species_results[q_sp][f"top{k}"] += hit

    # convert to percentages
    overall = {}
    for k in topk:
        overall[f"top{k}"] = 100.0 * results[k] / max(n_queries, 1)

    per_species = {}
    for sp, stats in species_results.items():
        cnt = max(stats["count"], 1)
        per_species[sp] = {
            "num_queries": stats["count"],
            **{f"top{k}": 100.0 * stats[f"top{k}"] / cnt for k in topk}
        }

    return {
        "overall": overall,
        "per_species": per_species,
        "num_queries": n_queries,
    }

In [ ]:
@torch.no_grad()
def evaluate_unknown_similarity(
    gallery_dict,
    unknown_query_dict,
):
    """
    Diagnostic only:
    For each unknown query, compute the max cosine similarity to same-species gallery.
    Lower is usually better separation from knowns.
    """
    gallery_feats = gallery_dict["features"]
    gallery_species = np.array(gallery_dict["datasets"])

    query_feats = unknown_query_dict["features"]
    query_species = np.array(unknown_query_dict["datasets"])

    species_to_gallery_idx = defaultdict(list)
    for i, sp in enumerate(gallery_species):
        species_to_gallery_idx[sp].append(i)

    max_sims = []
    per_species_sims = defaultdict(list)

    for q_idx in range(len(query_species)):
        q_feat = query_feats[q_idx:q_idx + 1]
        q_sp = query_species[q_idx]

        g_idx = species_to_gallery_idx[q_sp]
        if len(g_idx) == 0:
            continue

        g_feats = gallery_feats[np.array(g_idx)]
        sims = torch.matmul(q_feat, g_feats.T).squeeze(0)
        max_sim = float(torch.max(sims).item())

        max_sims.append(max_sim)
        per_species_sims[q_sp].append(max_sim)

    def summarize(arr):
        if len(arr) == 0:
            return {}
        arr = np.array(arr)
        return {
            "mean": float(arr.mean()),
            "std": float(arr.std()),
            "p50": float(np.percentile(arr, 50)),
            "p90": float(np.percentile(arr, 90)),
            "p95": float(np.percentile(arr, 95)),
            "p99": float(np.percentile(arr, 99)),
        }

    return {
        "overall": summarize(max_sims),
        "per_species": {sp: summarize(vals) for sp, vals in per_species_sims.items()},
        "num_queries": len(max_sims),
    }

In [ ]:
def train_one_epoch_sphereface2(
    model,
    loader,
    optimizer,
    device,
    epoch,
    scaler=None,
    scheduler=None,
    grad_clip_norm=None,
    log_interval=20,
    use_amp=True,
):
    model.train()

    loss_meter = AverageMeter("loss")
    top1_meter = AverageMeter("top1")
    top5_meter = AverageMeter("top5")
    top20_meter = AverageMeter("top20")
    batch_time_meter = AverageMeter("batch_time")
    data_time_meter = AverageMeter("data_time")
    grad_norm_meter = AverageMeter("grad_norm")

    start_epoch_time = time.time()
    end = time.time()

    progress = tqdm(loader, desc=f"Train Epoch {epoch}", leave=False)

    for step, batch in enumerate(progress):
        data_time = time.time() - end
        data_time_meter.update(data_time)

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if torch.any(labels < 0):
            raise ValueError("Training batch contains label < 0. Training set should not contain unknown identities.")

        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler is not None:
            with torch.cuda.amp.autocast():
                feats = model.forward_features(images)
                loss, logits = model.head.compute_loss(feats, labels, return_logits=True)

            scaler.scale(loss).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            grad_norm = compute_grad_norm(model.parameters())
            grad_norm_meter.update(grad_norm)

            scaler.step(optimizer)
            scaler.update()

        else:
            feats = model.forward_features(images)
            loss, logits = model.head.compute_loss(feats, labels, return_logits=True)

            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            grad_norm = compute_grad_norm(model.parameters())
            grad_norm_meter.update(grad_norm)

            optimizer.step()

        # metrics from SphereFace2 training logits
        topk = topk_accuracy_from_logits(logits.detach(), labels, topk=(1, 5, 20))
        bs = images.size(0)

        loss_meter.update(loss.item(), bs)
        top1_meter.update(topk[1], bs)
        top5_meter.update(topk[5], bs)
        top20_meter.update(topk[20], bs)

        batch_time = time.time() - end
        batch_time_meter.update(batch_time)
        end = time.time()

        if step % log_interval == 0 or step == len(loader) - 1:
            progress.set_postfix({
                "loss": f"{loss_meter.avg:.4f}",
                "top1": f"{top1_meter.avg:.2f}",
                "top5": f"{top5_meter.avg:.2f}",
                "top20": f"{top20_meter.avg:.2f}",
                "gnorm": f"{grad_norm_meter.avg:.2f}",
                "lr": format_lrs(optimizer),
            })

    if scheduler is not None:
        scheduler.step()

    epoch_time = time.time() - start_epoch_time

    return {
        "loss": loss_meter.avg,
        "top1": top1_meter.avg,
        "top5": top5_meter.avg,
        "top20": top20_meter.avg,
        "grad_norm": grad_norm_meter.avg,
        "batch_time": batch_time_meter.avg,
        "data_time": data_time_meter.avg,
        "epoch_time_sec": epoch_time,
        "lrs": get_optimizer_lrs(optimizer),
    }

In [ ]:
def validate_reid_sphereface2(
    model,
    train_eval_loader,
    val_known_loader,
    val_unknown_loader,
    device,
    compute_unknown_stats=True,
):
    # 1) gallery
    gallery = extract_features(model, train_eval_loader, device, desc="Gallery features")

    # 2) known queries
    known_queries = extract_features(model, val_known_loader, device, desc="Known val features")
    known_metrics = evaluate_known_retrieval_topk(
        gallery_dict=gallery,
        query_dict=known_queries,
        topk=(1, 5, 20),
    )

    # 3) unknown diagnostics
    unknown_metrics = None
    if compute_unknown_stats and val_unknown_loader is not None:
        unknown_queries = extract_features(model, val_unknown_loader, device, desc="Unknown val features")
        unknown_metrics = evaluate_unknown_similarity(
            gallery_dict=gallery,
            unknown_query_dict=unknown_queries,
        )

    return {
        "known": known_metrics,
        "unknown": unknown_metrics,
    }

In [ ]:
from dataclasses import dataclass
import os
import time
import json


@dataclass
class TrainConfig:
    epochs: int = 20
    use_amp: bool = True
    grad_clip_norm: float = 5.0
    log_interval: int = 20
    save_dir: str = "./checkpoints_sphereface2"
    save_best_only: bool = True
    monitor_metric: str = "val_top1"
    compute_unknown_stats: bool = True


def save_checkpoint(state, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(state, save_path)


def train_sphereface2_model(
    model,
    train_loader,
    train_eval_loader,
    val_known_loader,
    val_unknown_loader,
    optimizer,
    device,
    config: TrainConfig,
    scheduler=None,
    scaler=None,
    start_epoch=1,
    history=None,
    best_score=-float("inf"),
    best_epoch=-1,
):
    os.makedirs(config.save_dir, exist_ok=True)

    total_params, trainable_params = count_parameters(model)
    print("=" * 100)
    print("Training configuration")
    print("=" * 100)
    print(f"Device                : {device}")
    print(f"Epochs                : {config.epochs}")
    print(f"AMP                   : {config.use_amp}")
    print(f"Grad clip norm        : {config.grad_clip_norm}")
    print(f"Save dir              : {config.save_dir}")
    print(f"Monitor metric        : {config.monitor_metric}")
    print(f"Total params          : {total_params:,}")
    print(f"Trainable params      : {trainable_params:,}")
    print(f"Initial optimizer LRs : {format_lrs(optimizer)}")
    print("=" * 100)

    if history is None:
        history = []

    for epoch in range(start_epoch, config.epochs + 1):
        epoch_start = time.time()

        # if using PK batch sampler with epoch-dependent shuffling
        if hasattr(train_loader.batch_sampler, "set_epoch"):
            train_loader.batch_sampler.set_epoch(epoch)

        # -------------------------
        # Train
        # -------------------------
        train_stats = train_one_epoch_sphereface2(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            scaler=scaler,
            scheduler=scheduler,
            grad_clip_norm=config.grad_clip_norm,
            log_interval=config.log_interval,
            use_amp=config.use_amp,
        )

        # -------------------------
        # Validate
        # -------------------------
        # This function can stay the same because validation uses
        # model.forward_features(...) and retrieval, not CE logits.
        val_stats = validate_reid_sphereface2(
            model=model,
            train_eval_loader=train_eval_loader,
            val_known_loader=val_known_loader,
            val_unknown_loader=val_unknown_loader,
            device=device,
            compute_unknown_stats=config.compute_unknown_stats,
        )

        val_top1 = val_stats["known"]["overall"]["top1"]
        val_top5 = val_stats["known"]["overall"]["top5"]
        val_top20 = val_stats["known"]["overall"]["top20"]

        if config.monitor_metric == "val_top1":
            current_score = val_top1
        elif config.monitor_metric == "val_top5":
            current_score = val_top5
        elif config.monitor_metric == "val_top20":
            current_score = val_top20
        else:
            raise ValueError(f"Unknown monitor metric: {config.monitor_metric}")

        is_best = current_score > best_score
        if is_best:
            best_score = current_score
            best_epoch = epoch

        epoch_time = time.time() - epoch_start

        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_top1": train_stats["top1"],
            "train_top5": train_stats["top5"],
            "train_top20": train_stats["top20"],
            "train_grad_norm": train_stats["grad_norm"],
            "val_top1": val_top1,
            "val_top5": val_top5,
            "val_top20": val_top20,
            "epoch_time_sec": epoch_time,
            "best_score_so_far": best_score,
            "best_epoch_so_far": best_epoch,
        }
        history.append(row)

        print("\n" + "=" * 120)
        print(f"Epoch {epoch}/{config.epochs}")
        print("=" * 120)
        print("TRAIN")
        print(f"  Loss                : {train_stats['loss']:.5f}")
        print(f"  Top-1               : {train_stats['top1']:.2f}")
        print(f"  Top-5               : {train_stats['top5']:.2f}")
        print(f"  Top-20              : {train_stats['top20']:.2f}")
        print(f"  Grad norm           : {train_stats['grad_norm']:.3f}")
        print(f"  Avg data time       : {train_stats['data_time']:.3f} s")
        print(f"  Avg batch time      : {train_stats['batch_time']:.3f} s")
        print(f"  Optimizer LRs       : {format_lrs(optimizer)}")

        print("\nVALIDATION (Known retrieval against train gallery)")
        print(f"  Top-1               : {val_top1:.2f}")
        print(f"  Top-5               : {val_top5:.2f}")
        print(f"  Top-20              : {val_top20:.2f}")

        print("\nPer-species known retrieval")
        for sp, stats in val_stats["known"]["per_species"].items():
            print(
                f"  {sp:18s} | "
                f"queries={stats['num_queries']:4d} | "
                f"top1={stats['top1']:.2f} | "
                f"top5={stats['top5']:.2f} | "
                f"top20={stats['top20']:.2f}"
            )

        if val_stats["unknown"] is not None:
            print("\nUNKNOWN diagnostics (max similarity to same-species train gallery)")
            overall_u = val_stats["unknown"]["overall"]
            if len(overall_u) > 0:
                print(
                    f"  mean={overall_u['mean']:.4f} | std={overall_u['std']:.4f} | "
                    f"p50={overall_u['p50']:.4f} | p90={overall_u['p90']:.4f} | "
                    f"p95={overall_u['p95']:.4f} | p99={overall_u['p99']:.4f}"
                )
                for sp, stats in val_stats["unknown"]["per_species"].items():
                    print(
                        f"  {sp:18s} | "
                        f"mean={stats['mean']:.4f} | p90={stats['p90']:.4f} | "
                        f"p95={stats['p95']:.4f} | p99={stats['p99']:.4f}"
                    )

        print("\nCHECKPOINT STATUS")
        print(f"  Current monitored score ({config.monitor_metric}) : {current_score:.2f}")
        print(f"  Best score so far                                : {best_score:.2f}")
        print(f"  Best epoch so far                                : {best_epoch}")
        print(f"  Epoch time                                       : {epoch_time:.1f} s")
        print("=" * 120)

        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "best_score": best_score,
            "best_epoch": best_epoch,
            "config": config.__dict__,
        }

        latest_path = os.path.join(config.save_dir, "last.pt")
        save_checkpoint(ckpt, latest_path)

        if is_best:
            best_path = os.path.join(config.save_dir, "best.pt")
            save_checkpoint(ckpt, best_path)

        if not config.save_best_only:
            epoch_path = os.path.join(config.save_dir, f"epoch_{epoch:03d}.pt")
            save_checkpoint(ckpt, epoch_path)

        with open(os.path.join(config.save_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

    print("\nTraining finished.")
    print(f"Best {config.monitor_metric}: {best_score:.2f} at epoch {best_epoch}")

    return history

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((440, 440)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_eval_dataset = OpenSetAnimalDataset(
    csv_file="fine_tune_train.csv",
    root_dir="dataset_root",
    transform=eval_transform,
    preprocess_fn=default_species_preprocess,  
    label_to_idx=train_dataset.label_to_idx,
    training=False,
    use_proc_path=False,
    return_path=True,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

criterion = nn.CrossEntropyLoss()

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

config = TrainConfig(
    epochs=20,
    use_amp=True,
    grad_clip_norm=5.0,
    log_interval=20,
    save_dir="./checkpoints_miewid_sphereface2",
    save_best_only=True,
    monitor_metric="val_top1",
    compute_unknown_stats=True,
)

history = train_sphereface2_model(
    model=model,
    train_loader=train_loader,
    train_eval_loader=train_eval_loader,
    val_known_loader=val_known_loader,
    val_unknown_loader=val_unknown_loader,
    optimizer=optimizer,
    # criterion=criterion,
    device=device,
    config=config,
    scheduler=None,   
    scaler=scaler,
)